# ERP DataProtect - Serveur Ollama (Mistral 7B Fine-tune)
Executer les cellules dans l'ordre. Copier l'URL .lhr.life de la Cellule 7 dans docker-compose.yml

In [ ]:
# CELLULE 1 - Verification GPU
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if result.returncode == 0:
    print(result.stdout)
    print('OK GPU disponible')
else:
    raise SystemExit('GPU requis - Aller dans Runtime -> T4 GPU')

In [ ]:
# CELLULE 2 - Installation Ollama
import subprocess
print('Installation Ollama...')
result = subprocess.run('curl -fsSL https://ollama.com/install.sh | sh', shell=True, capture_output=True, text=True)
if result.returncode == 0:
    print('OK Ollama installe')
else:
    print(result.stderr)
    raise SystemExit('Echec installation Ollama')

In [ ]:
# CELLULE 3 - Montage Google Drive
from google.colab import drive
import os
drive.mount('/content/drive')
GGUF_PATH = '/content/drive/MyDrive/ERP_DATAPROTECT/mistral-7b-instruct-v0.3.Q4_K_M.gguf'
if os.path.exists(GGUF_PATH):
    size_gb = os.path.getsize(GGUF_PATH) / (1024**3)
    print(f'OK GGUF trouve: {size_gb:.2f} GB')
else:
    raise SystemExit(f'GGUF introuvable: {GGUF_PATH}')

In [ ]:
# CELLULE 4 - Lancement Ollama sur 0.0.0.0 (obligatoire pour le tunnel)
import subprocess, time, requests, os
print('Demarrage Ollama sur 0.0.0.0...')
subprocess.run(['pkill', '-9', 'ollama'], capture_output=True)
time.sleep(2)
env = os.environ.copy()
env['OLLAMA_HOST'] = '0.0.0.0:11434'
subprocess.Popen(['ollama', 'serve'], env=env, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
for i in range(30):
    try:
        r = requests.get('http://localhost:11434/api/tags', timeout=2)
        if r.status_code == 200:
            print(f'OK Ollama pret apres {i+1}s')
            break
    except:
        pass
    time.sleep(1)
    print(f'  ... {i+1}s')

In [ ]:
# CELLULE 5 - Creation du modele erp-dataprotect
import subprocess
GGUF_PATH = '/content/drive/MyDrive/ERP_DATAPROTECT/mistral-7b-instruct-v0.3.Q4_K_M.gguf'
modelfile = f"""FROM {GGUF_PATH}
TEMPLATE \"\"\"[INST] {{{{ .Prompt }}}} [/INST]\"\"\"
PARAMETER temperature 0.7
PARAMETER top_p 0.95
PARAMETER top_k 40
PARAMETER repeat_penalty 1.1
PARAMETER num_ctx 2048
PARAMETER stop \"[INST]\"
PARAMETER stop \"[/INST]\"
PARAMETER stop \"</s>\"
SYSTEM \"\"\"Tu es l assistant IA de l ERP DataProtect. Tu reponds aux questions des employes concernant les modules RH Finance IT et Operations. Tu bases tes reponses sur la documentation ERP fournie dans le contexte. Reponds en francais de facon concise et professionnelle.\"\"\"
"""
with open('/tmp/Modelfile', 'w') as f:
    f.write(modelfile)
print('Creation du modele erp-dataprotect...')
result = subprocess.run(['ollama', 'create', 'erp-dataprotect', '-f', '/tmp/Modelfile'], capture_output=True, text=True)
if result.returncode == 0:
    print('OK Modele cree')
else:
    print(result.stderr)
    raise SystemExit('Echec creation modele')
print(subprocess.run(['ollama', 'list'], capture_output=True, text=True).stdout)

In [ ]:
# CELLULE 6 - Test du modele (pre-chargement VRAM)
import requests
print('Test modele - chargement VRAM (30s)...')
r = requests.post('http://localhost:11434/api/generate',
    json={'model': 'erp-dataprotect', 'prompt': 'Bonjour', 'stream': False, 'options': {'num_predict': 50}},
    timeout=120)
if r.status_code == 200:
    print('OK Modele repond:')
    print(r.json()['response'])
else:
    print(f'ERREUR: {r.status_code} - {r.text}')

In [ ]:
# CELLULE 7 - Tunnel localhost.run avec cle SSH
import subprocess, re
print('Lancement localhost.run tunnel...')
process = subprocess.Popen(
    ['ssh', '-o', 'StrictHostKeyChecking=no', '-i', '/root/.ssh/id_rsa', '-R', '80:localhost:11434', 'localhost.run'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
)
for line in iter(process.stdout.readline, ''):
    print(line.strip())
    if '.lhr.life' in line:
        match = re.search(r'https://[\w]+\.lhr\.life', line)
        if match:
            url = match.group(0)
            print(f'\n{"="*60}')
            print('TUNNEL ACTIF')
            print(f'URL: {url}')
            print(f'COPIER dans docker-compose.yml:')
            print(f'  - OLLAMA_URL={url}')
            print(f'{"="*60}')
            break

In [ ]:
# CELLULE 8 - Maintien session anti-timeout
import time, requests, subprocess, os
from datetime import datetime
print('Surveillance Ollama toutes les 5 minutes - Ne pas interrompre\n')

def restart_ollama():
    subprocess.run(['pkill', '-9', 'ollama'], capture_output=True)
    time.sleep(2)
    env = os.environ.copy()
    env['OLLAMA_HOST'] = '0.0.0.0:11434'
    subprocess.Popen(['ollama', 'serve'], env=env, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    time.sleep(5)
    subprocess.run(['ollama', 'create', 'erp-dataprotect', '-f', '/tmp/Modelfile'], capture_output=True)
    print(f'[{datetime.now().strftime("%H:%M:%S")}] Ollama redémarre')

while True:
    try:
        r = requests.get('http://localhost:11434/api/tags', timeout=5)
        now = datetime.now().strftime('%H:%M:%S')
        if r.status_code == 200:
            models = [m['name'] for m in r.json().get('models', [])]
            print(f'[{now}] OK Ollama actif | Modeles: {models}')
        else:
            restart_ollama()
    except:
        restart_ollama()
    time.sleep(300)